# 🌌 Phase 4 — HPO Ensemble & Evaluation (Quick Tour)

Welcome! This cell runs the **Phase 4 pipeline**: it harvests champions, loads time series, builds ensembles, and renders clean visuals (plus an optional risk CDF panel). Here’s the friendly map of what happens 👇


## 🧭 What this program does 

**Turns raw experiment results into trustworthy, aligned, and calibrated ensemble insights**—with plots, metrics, and optional risk curves—so you can make decisions quickly and confidently.

---

## 🧩 Inputs & Knobs (you can tweak)

* 🎯 **Filters:** `FILTERS = {"dataset": "VOLVE", "well": "15/9-F-14"}`
* 🗂 **Paths:** `PROJECT_ROOT`, `SERIES_STORE_ROOT`
* 🎨 **Palette:** `"default"`
* 📈 **Risk Panel:** `ENABLE_RISK_PANEL`, `RISK_SELECTOR`, `RISK_HORIZONS`, `RISK_WEIGHTING`, `RISK_TEMP`, `RISK_ARCHES`
* 🧪 **Scoring & selection:** set in `Phase4Config` (e.g., `scoring_strategy="weighted_score"`, `metric_weights`, `posthoc_overrides`)

---

## 🛠️ Pipeline Stages (at a glance)

### 🧪 1) Champion Harvest

Picks top-performing models per **well × architecture** using your scoring strategy.

* Uses **weighted** or **robust** score with **lower-is-better** semantics.
* Optional Pareto/MAD guards via `posthoc_overrides`.

### 📦 2) Series Store Loader

Reads the champion **time series** (Parquet) + merges **boundaries** and **full history**.
Everything is aligned to a single **t-axis** for clean plotting.

### ✂️ 2.5) Train Restriction

Keeps **only one primary job** per well in **TRAIN**, leaving **VAL/TEST** intact—cleaner visuals and faster rendering.

### 🧬 3) Ensemble Builder

* **Intra-family** means (e.g., `seq2`, `arps`)
* **Inter-family** final ensemble (meta-mean)
* Annotates **n_members** for transparency.

### 🎨 4) Visualization & Reporting

Publishes **conjugated plots** (Inter + Intra) and **members spaghetti** per family.
Shows **Train/Val/Test** regions, ground truth, ensemble means, and uncertainty.

---

## 📊 (Optional) Risk Mini-Panel

If `ENABLE_RISK_PANEL = True`, you’ll see a **compact CDF** per selected **architecture** and **horizon**:

* Selector: `"train" | "val" | "test" | "val+test"`
* Horizons: e.g., `300, 600, 900, -1` (where `-1` means “until the end”)
* Weights: `"uniform"` or `"distance_softmax"` (temperature `RISK_TEMP`)

---

## 🧾 What you get back (artifacts)

* `champions_df` — selected models (post-hoc filters applied)
* `series_df` — aligned champion series (with splits)
* `intra_family_df` — per-architecture ensemble
* `final_ensemble_df` — meta-ensemble (Inter)
* `boundaries_df` — manifest (train/val/test)
* `full_history_by_well` — ground-truth map per well

Plus: quick storage sanity via `check_store_health(...)`.

---

## 🧠 Pro tips

* Change `FILTERS["well"]` to quickly pivot wells without regenerating configs.
* Keep palettes consistent across plots via `PALETTE`.
* Turn on `log_mode="compact"` for clean notebook logs, `"verbose"` for deep dives.
* Use `enable_risk_plots=False` in `Phase4Config` to keep risk computation in-memory off (Stage 4.1); use the **quick panel** instead for instant visuals.

---

## 🚀 One-line mental model

**Leaderboards → (Score & Select) → (Load & Align) → (Restrict Train) → (Ensemble) → (Plot & Risk CDF).**


In [ ]:
# %% Phase 4 — Single Entry Point (standard / replay / compare)
from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional
import pandas as pd

from common.phase_orchestrator import (
    Phase4Config,
    run_phase4_pipeline,
    run_phase4_compare,
)
from utils.utilities import check_store_health, quick_risk_panel

# =========================
# 1) User-facing knobs
# =========================
FILTERS: Dict[str, object] = {
    "dataset": "INISIM_IV",
    # "well": "15/9-F-14",
    "well": "P15",
}

CAMPAIGN = "HPO_153_Lag_100_Horizon_150"

PROJECT_ROOT = Path("/home/gabriel/Documentos/Equinor")
SERIES_STORE_ROOT = PROJECT_ROOT / "series_store"
PALETTE = "default"

# =========================
# 2) Execution mode
# =========================
# Options:
#   "standard"        -> legacy pipeline
#   "replay_global"   -> replay with one global policy
#   "replay_selected" -> replay using selected_anchor_policies.csv
#   "compare"         -> run both arms side by side
RUN_MODE = "replay_selected"

# --- replay_global knobs ---
REPLAY_POLICY_NAME = "trend_strict"
REPLAY_POLICY_OVERRIDES: Optional[Dict[str, object]] = None

# --- replay_selected knobs ---
SELECTED_POLICY_PATH = (
    PROJECT_ROOT / "artifacts" / "anchor_ablation_dev" / "selected_anchor_policies.csv"
)
DEFAULT_POLICY_NAME = None  # e.g. "baseline_default" if some well is missing from the CSV

# --- compare / display knobs ---
SHOW_STANDARD_PLOTS = False
SHOW_REPLAY_PLOTS = True

# Which arm should feed the quick risk panel?
#   "standard" | "replay"
PANEL_ARM = "replay"

# =========================
# 3) Risk panel knobs
# =========================
ENABLE_RISK_PANEL = True
RISK_SELECTOR     = "val+test"          # "train" | "val" | "test" | "val+test"
RISK_HORIZONS     = [-1]                # e.g. [300, 600, 900, -1]
RISK_WEIGHTING    = "uniform"           # "uniform" | "distance_softmax"
RISK_TEMP         = 0.5
RISK_ARCHES       = ("seq2", "arps")

# =========================
# 4) Selection / ensemble knobs
# =========================
ENSEMBLES = 30

# =========================
# 5) Build config
# =========================
cfg = Phase4Config(
    # --- Scoring & selection ---
    scoring_strategy="weighted_score",
    campaigns_to_ensemble={"T_current": CAMPAIGN},
    n_champions_per_group=2,
    metric_weights={
        "val_smape_agg": 5.0,
        "val_smape_cum": 1.0,
    },
    lower_is_better={
        "val_smape_agg": True,
        "val_smape_cum": True,
        "weighted_score": True,
        "robust_score": True,
    },
    posthoc_overrides=dict(
        top_strategies_per_well=ENSEMBLES,
        per_strategy_k=50,
        selection_strategy="best_of_the_best",
        apply_pareto=False,
        mad_guard={
            "enabled": True,
            "alpha": 0.5,
            "metrics": ["val_smape_cum", "val_smape_agg"],
            "log": True,
            "side": "right",
        },
        valcum_gate={"q_low": 0.1, "q_high": 0.9},
    ),

    # --- Scope ---
    wells_to_analyze=[
        "P11", "P12", "P13", "P14", "P15", "P16",
        "15/9-F-12", "15/9-F-14",
    ],

    # --- Paths ---
    project_root=PROJECT_ROOT,
    series_store_root=SERIES_STORE_ROOT,

    # --- Visuals ---
    palette=PALETTE,
    enable_plots=True,
    show_family_traces=False,
    show_champion_traces=False,

    # --- Logging ---
    log_mode="compact",
    log_width=100,

    # --- Risk (Stage 4.1) ---
    enable_risk_plots=False,            # quick panel remains available below
    risk_splits=[RISK_SELECTOR],
    risk_horizons_days=RISK_HORIZONS,
    risk_weighting=RISK_WEIGHTING,
    risk_distance_temp=RISK_TEMP,
    risk_palette=PALETTE,
    risk_show_tables=True,
)

# =========================
# 6) Small notebook helpers
# =========================
def _unpack_artifacts(tag: str, art: Dict[str, pd.DataFrame]) -> None:
    print(f"\n[{tag}] artifacts keys:")
    print(sorted(art.keys()))

    final_df = art.get("final_ensemble_df", pd.DataFrame())
    intra_df = art.get("intra_family_df", pd.DataFrame())
    series_df = art.get("series_df", pd.DataFrame())
    risk_df = art.get("risk_df", pd.DataFrame())

    print(f"[{tag}] series_df shape:         {getattr(series_df, 'shape', None)}")
    print(f"[{tag}] intra_family_df shape:  {getattr(intra_df, 'shape', None)}")
    print(f"[{tag}] final_ensemble_df shape:{getattr(final_df, 'shape', None)}")
    print(f"[{tag}] risk_df shape:          {getattr(risk_df, 'shape', None)}")

    anchor_cfg = art.get("anchor_config", None)
    if anchor_cfg is not None:
        print(f"[{tag}] anchor_config: {anchor_cfg}")

    if "selected_policy_source" in art:
        print(f"[{tag}] selected_policy_source: {art.get('selected_policy_source')}")


def _show_health(tag: str, art: Dict[str, pd.DataFrame]) -> None:
    print(f"\n===== STORE HEALTH: {tag} =====")
    check_store_health(
        art,
        series_store_root=SERIES_STORE_ROOT,
        max_show=10,
    )


def _show_risk_panel(tag: str, art: Dict[str, pd.DataFrame]) -> None:
    if not ENABLE_RISK_PANEL:
        return

    well_for_panel = str(FILTERS.get("well") or cfg.wells_to_analyze[0])
    print(f"\n===== QUICK RISK PANEL: {tag} | well={well_for_panel} =====")
    try:
        quick_risk_panel(
            artifacts=art,
            well=well_for_panel,
            arch_list=RISK_ARCHES,
            selector=RISK_SELECTOR,
            horizons=RISK_HORIZONS,
            weighting=RISK_WEIGHTING,
            temp=RISK_TEMP,
            palette=PALETTE,
            show=True,
        )
    except Exception as e:
        print(f"[quick_risk_panel:{tag}] skipped: {e}")


# =========================
# 7) Run
# =========================
artifacts = None
artifacts_standard = None
artifacts_replay = None
compare_bundle = None

if RUN_MODE == "standard":
    artifacts = run_phase4_pipeline(cfg, filters=FILTERS)

elif RUN_MODE == "replay_global":
    compare_bundle = run_phase4_compare(
        cfg,
        filters=FILTERS,
        replay_policy_name=REPLAY_POLICY_NAME,
        replay_policy_overrides=REPLAY_POLICY_OVERRIDES,
        show_standard_plots=SHOW_STANDARD_PLOTS,
        show_replay_plots=SHOW_REPLAY_PLOTS,
    )
    artifacts_standard = compare_bundle["standard"]
    artifacts_replay = compare_bundle["replay"]
    artifacts = artifacts_replay

elif RUN_MODE == "replay_selected":
    compare_bundle = run_phase4_compare(
        cfg,
        filters=FILTERS,
        selected_policy_source=SELECTED_POLICY_PATH,
        default_policy_name=DEFAULT_POLICY_NAME,
        show_standard_plots=SHOW_STANDARD_PLOTS,
        show_replay_plots=SHOW_REPLAY_PLOTS,
    )
    artifacts_standard = compare_bundle["standard"]
    artifacts_replay = compare_bundle["replay"]
    artifacts = artifacts_replay

elif RUN_MODE == "compare":
    if SELECTED_POLICY_PATH is not None and Path(SELECTED_POLICY_PATH).exists():
        compare_bundle = run_phase4_compare(
            cfg,
            filters=FILTERS,
            selected_policy_source=SELECTED_POLICY_PATH,
            default_policy_name=DEFAULT_POLICY_NAME,
            show_standard_plots=SHOW_STANDARD_PLOTS,
            show_replay_plots=SHOW_REPLAY_PLOTS,
        )
    else:
        compare_bundle = run_phase4_compare(
            cfg,
            filters=FILTERS,
            replay_policy_name=REPLAY_POLICY_NAME,
            replay_policy_overrides=REPLAY_POLICY_OVERRIDES,
            show_standard_plots=SHOW_STANDARD_PLOTS,
            show_replay_plots=SHOW_REPLAY_PLOTS,
        )

    artifacts_standard = compare_bundle["standard"]
    artifacts_replay = compare_bundle["replay"]
    artifacts = artifacts_replay

else:
    raise ValueError(
        f"Unknown RUN_MODE='{RUN_MODE}'. "
        "Use one of: 'standard', 'replay_global', 'replay_selected', 'compare'."
    )

# =========================
# 8) Convenience unpack
# =========================
champions_df         = artifacts["champions_df"]
series_df            = artifacts["series_df"]
intra_family_df      = artifacts["intra_family_df"]
final_ensemble_df    = artifacts["final_ensemble_df"]
boundaries_df        = artifacts["boundaries_df"]
full_history_by_well = artifacts["full_history_by_well"]

# =========================
# 9) Quick structural debug
# =========================
print(f"\nRUN_MODE = {RUN_MODE}")
if compare_bundle is not None:
    print("compare_bundle keys:", compare_bundle.keys())
    if "replay_mode" in compare_bundle:
        print("replay_mode:", compare_bundle["replay_mode"])

_unpack_artifacts("ACTIVE", artifacts)

if artifacts_standard is not None:
    _unpack_artifacts("STANDARD", artifacts_standard)

if artifacts_replay is not None:
    _unpack_artifacts("REPLAY", artifacts_replay)

# =========================
# 10) Store health
# =========================
_show_health("ACTIVE", artifacts)

if RUN_MODE == "compare":
    if artifacts_standard is not None:
        _show_health("STANDARD", artifacts_standard)
    if artifacts_replay is not None:
        _show_health("REPLAY", artifacts_replay)

# =========================
# 11) Optional quick risk panel
# =========================
if RUN_MODE == "compare":
    if PANEL_ARM == "standard" and artifacts_standard is not None:
        _show_risk_panel("STANDARD", artifacts_standard)
    elif PANEL_ARM == "replay" and artifacts_replay is not None:
        _show_risk_panel("REPLAY", artifacts_replay)
else:
    _show_risk_panel("ACTIVE", artifacts)

In [ ]:
import re
from pathlib import Path
import shutil

arquivo = Path("/home/gabriel/Documentos/Concurso_2026/14_Linguagem_SQL.md")  # troque pelo nome do seu arquivo
backup = arquivo.with_suffix(arquivo.suffix + ".bak")

shutil.copy2(arquivo, backup)

texto = arquivo.read_text(encoding="utf-8")

# Preserva blocos de código ```...```
partes = re.split(r"(```.*?```)", texto, flags=re.DOTALL)

# Fora dos blocos, troca `algo` por **algo**
for i in range(len(partes)):
    if not partes[i].startswith("```"):
        partes[i] = re.sub(r"`([^`\n]+)`", r"**\1**", partes[i])

novo_texto = "".join(partes)

arquivo.write_text(novo_texto, encoding="utf-8")

print(f"Patch aplicado em: {arquivo}")
print(f"Backup criado em: {backup}")